# Approximation of the solution of pre-defined physical models in `Scimba`

In this first tutorial, we will walk you through the basics of `scimba_jax`
for defining and training Physics-Informed Neural Networks (PINNs)
for two toy examples, a stationary one and a time-dependent one.

### `Scimba` and `jax` setting

`Scimba_jax` is based on `jax` for tensor arithmetic, auto-differentiation, and for defining and evaluating neural networks.

At initialization, `Scimba` sets `jax` default floating point format to `double`.
We discourage using `float32` as the default type (Natural Gradient descent, the default optimizer in `Scimba_jax`, requires high precision).


In [ ]:
# # uncomment the line below if you are on Google Colab
# !pip install scimba

import jax
import jax.numpy as jnp

import scimba_jax

See version, floating arithmetic precision and current device with:


In [ ]:
scimba_jax.set_verbosity(True)
scimba_jax.set_verbosity(False)

The device can be checked and changed with the `jax` API:


In [ ]:
a = jnp.ones(5)
print("default device: ", a.device)

# # uncomment the following lines to make sure JAX uses a GPU (errors if no GPU is available)
# jax.config.update("jax_default_device", jax.devices("gpu")[0])
# a = jnp.ones(5)
# print("default device: ", a.device)

## 1/ Approximation of the solution of a parametric 2D Laplacian

We consider the equation:

$$- \mu \Delta u = 2(\mu\pi)^2 \sin(\pi x) \sin(\pi y)$$

on $(x,y) \in \Omega := [-1, 1]\times[-1, 1]$ with parameter $\mu\in \mathbb{M}=[1,2]$ subject to the Dirichlet condition:

$$ u = 0 $$

on $\partial\Omega$.

The exact solution is given by

$$ u(x,y,\mu) = \mu \sin(\pi x) \sin(\pi y). $$

Hence, we are essentially solving a **three-dimensional problem**, with two space dimensions and one parameter dimension. The solution is a function of three variables, $u(x,y,\mu)$. We emphasize that we solve this problem *in a monolithic fashion*, without snapshots on $\mu$, on the whole continuous interval $\mu \in \mathbb{M}=[1,2]$.


### Geometric domain $\Omega \subset \mathbb{R}^2$

In `Scimba`, in a $d$-dimensional ambient space, $d$-dimensional geometric domains are objects of class `VolumetricDomain`, and $(d-1)$-dimensional geometric domains are objects of class `SurfacicDomain`.

The most common types of domains (Cartesian products of intervals, disks, ...) are implemented in the modules `scimba_jax.domains.meshless_domains.domains_1d` and `scimba_jax.domains.meshless_domains.domains_2d`.

Here, we define $\Omega := [-1, 1]\times[-1, 1]$ with:


In [ ]:
from scimba_jax.domains.meshless_domains.domains_2d import Square2D

domain_x = Square2D(
    [(-1.0, 1.0), (-1.0, 1.0)],
    is_main_domain=True,
)

The keyword argument `is_main_domain` indicates whether the created object
is a main domain or is a subdomain of a main domain; `Scimba` allows one to build complex geometries by combining holes and subdomains.

### Sampler on $\Omega\times \mathbb{M}$

The next step is to define samplers for the geometric and parameter domains. The module `scimba_jax.nonlinear_approximation.integration` provides uniform samplers for physical and parameter domains.


In [ ]:
from scimba_jax.nonlinear_approximation.integration.monte_carlo import (
    DomainSampler,
    TensorizedSampler,
)
from scimba_jax.nonlinear_approximation.integration.monte_carlo_parameters import (
    UniformParametricSampler,
)

domain_mu = [(1.0, 2.0)]
sampler = TensorizedSampler(
    [DomainSampler(domain_x), UniformParametricSampler(domain_mu)],
    bc=True,
)

A tensorized sampler gathers samplers for the domains of the variables of the problem.
Notice the optional argument `bc=True` which tells the sampler to also sample the boundary of the geometric domain.


### Right-hand side and exact solution

Next we define the right-hand side of the residual and, since we know it, the analytical expression of the solution. We will use it to assess the accuracy of the computed approximate solution.


In [ ]:
def f_rhs(xy: jnp.ndarray, mu: jnp.ndarray) -> jnp.ndarray:
    """Right-hand side of the PDE."""
    x, y = xy[..., 0:1], xy[..., 1:2]
    return 2 * (mu[..., 0:1] * jnp.pi) ** 2 * jnp.sin(jnp.pi * x) * jnp.sin(jnp.pi * y)


def analytic_solution(xy: jnp.ndarray, mu: jnp.ndarray) -> jnp.ndarray:
    """Exact solution of the PDE."""
    x, y = xy[..., 0:1], xy[..., 1:2]
    return mu * jnp.sin(jnp.pi * x) * jnp.sin(jnp.pi * y)

### Physical model

Several PDEs are already implemented in `Scimba`; here we use the 2D Laplacian with Dirichlet boundary conditions:


In [ ]:
from scimba_jax.physical_models.elliptic_pde.laplacians import (
    ParametricLaplacianDirichletND,
)

model = ParametricLaplacianDirichletND(
    domain_x,
    f_rhs,
    bc="weak",
)

One can also create a custom model, as described in the next tutorial.

Here the right-hand side of the Dirichlet boundary condition is the zero function; when it is not the case, one must define it and pass it through the named argument `f_bc_rhs` when constructing the model.

In `scimba_jax`, the right-hand side of a boundary condition residual also takes the normals as input.


In [ ]:
def f_bc_rhs(xy: jnp.ndarray, n: jnp.ndarray, mu: jnp.ndarray) -> jnp.ndarray:
    """Right-hand side of the Dirichlet boundary condition.

    Optional here as homogeneous Dirichlet boundary conditions
    are the default.
    """
    x = xy[..., 0:1]
    return jnp.zeros_like(x)


model = ParametricLaplacianDirichletND(domain_x, f_rhs, bc="weak", f_bc_rhs=f_bc_rhs)

### Approximation space

In `Scimba`, the solution of a PDE is approximated by an element $u_{\theta}$ of a set of parameterized functions called **approximation space**.

Here we wish to approximate the solution with a multi-layer perceptron (MLP) with 3 scalar inputs ($x$, $y$ and $\mu$), 1 output ($u(x,y,\mu)$) and two hidden layers of $16$ neurons each:


In [ ]:
from scimba_jax.nonlinear_approximation.networks.mlp import MLP
from scimba_jax.nonlinear_approximation.approximation_spaces.approximation_spaces import (
    ApproximationSpace,
)

key = jax.random.PRNGKey(0)  # initialize the random number generator

INPUT_SIZE = 3
OUTPUT_SIZE = 1  # scalar function u
HIDDEN_SIZES = [16, 16]  # two hidden layers with 16 neurons each

neural_network = MLP(
    in_size=INPUT_SIZE, out_size=OUTPUT_SIZE, hidden_sizes=HIDDEN_SIZES, key=key
)
space = ApproximationSpace(
    {"x": 2, "mu": 1},  # x has dim 2, mu has dim 1
    [(neural_network, "scalar", None)],  # neural_network models a scalar function u
    model_type="x_mu",  # u depends on x and mu
)

### `Scimba` PINN

We finally create a PINN as an instance of the class `Projector`:


In [ ]:
from scimba_jax.nonlinear_approximation.numerical_solvers.projectors import Projector

pinn = Projector(model, space, sampler, matrix_regularization=1e-5)

The default optimizer used in `scimba_jax` PINNs is Natural Gradient descent (see below);
the optional argument `matrix_regularization` is an important parameter used in this optimizer;
by default, it takes value `1e-6`.

The PINN is not trained yet; let us sample some collocation points and evaluate the loss functions associated with the interior residual and the boundary residual of the model:


In [ ]:
N_COLLOC = 1000  # the number of collocation points for the interior residual
N_BC_COLLOC = 1000  # the number of collocation points for the boundary residual
key, sample_dict = sampler.sample(key, N_COLLOC, N_BC_COLLOC)

losses = pinn.evaluate_losses(space, sample_dict)

print("interior loss before training: ", losses["interior"])
print("boundary loss before training: ", losses["boundary"])
print("total    loss before training: ", losses["total"])

The total loss is a weighted sum of the losses.
By default the weights are:
- $1$ for the interior residual
- $10$ for the boundary residual (and the initial residual for time-dependent problems)

We will show later how to adjust those weights.

### How to train the PINN:

By default, `scimba_jax` PINNs are trained with [Energy Natural Gradient](https://arxiv.org/pdf/2302.13163) preconditioned gradient descent:


In [ ]:
import timeit

N_EPOCHS = 50

start = timeit.default_timer()
key, pinn = pinn.project(key, space, N_EPOCHS, N_COLLOC, N_BC_COLLOC)
end = timeit.default_timer()

print(f"time for {N_EPOCHS} epochs: {end - start:.2f} s")

print("interior loss after training: ", pinn.best_loss["interior"])
print("boundary loss after training: ", pinn.best_loss["boundary"])
print("total    loss after training: ", pinn.best_loss["total"])

the training loop returns:
- a state for the random generator (`key`),
- a modified projector; the trained approximation space is accessed through `pinn.space`.

### Plot the trained PINN and the evolution of the loss functions:
Graphical output for a single parameter value ($\mu = 1.5$, in the middle of the interval $\mathbb{M}=[1,2]$):


In [ ]:
import matplotlib.pyplot as plt

from scimba_jax.plots.plots_nd import plot_abstract_approx_space

plot_abstract_approx_space(
    pinn.space,  # the approximation space
    domain_x,  # the spatial domain
    domain_mu,  # the parameter's domain
    loss=pinn.losses,  # for plot of the loss: the losses
    residual=pinn.model,  # for plot of the residual: the pde
    error=analytic_solution,  # for plot of the error with respect to a func: the func
    draw_contours=True,
    n_drawn_contours=20,
)

Graphical output for three random parameter values in the interval $\mathbb{M}=[1,2]$:


In [ ]:
for _ in range(3):
    plot_abstract_approx_space(
        pinn.space,  # the approximation space
        domain_x,  # the spatial domain
        domain_mu,  # the parameter's domain
        loss=pinn.losses,  # for plot of the loss: the losses
        residual=pinn.model,  # for plot of the residual: the pde
        error=analytic_solution,  # for plot of the error with respect to a function
        draw_contours=True,
        n_drawn_contours=20,
        parameters_values="random",
    )

## 2/ Playing with the hyperparameters of the PINN

The error of a PINN is usually decomposed into three contributions:

- the **approximation error**: is the approximation space expressive enough to represent
  the exact solution? It is set by the number of degrees of freedom of the network,
  that is by the width and depth of its hidden layers (2.1 below);
- the **discretization error**: the integrals in the loss are approximated by Monte
  Carlo sums over a finite number of collocation points; too few points make the
  discrete loss a poor approximation of the continuous one (2.2 below);
- the **optimization error**: the loss is never minimized exactly; the remaining gap
  depends on the optimizer and the number of epochs (2.3), on the weights of the
  residuals in the loss (2.4), and on the matrix regularization of the natural
  gradient optimizers (2.5).

In each sub-section below, change the proposed hyperparameter, train a new PINN, and
compare it with the reference PINN `pinn` of section 1 (same network, same
collocation points, default optimizer, weights and matrix regularization). The error
component that decreases is the one this hyperparameter controls.


In [ ]:
from scimba_jax.plots.plots_nd import plot_abstract_approx_spaces


def n_dofs(space) -> int:
    """Number of trainable parameters (degrees of freedom) of an approximation space."""
    leaves = jax.tree_util.tree_leaves(space)
    return sum(int(leaf.size) for leaf in leaves if hasattr(leaf, "size"))


def train_and_report(key, pinn, space, n_epochs, n_colloc, n_bc_colloc, label=""):
    """Train a PINN, then print the training time and the final losses."""
    start = timeit.default_timer()
    key, pinn = pinn.project(key, space, n_epochs, n_colloc, n_bc_colloc)
    end = timeit.default_timer()
    print(f"[{label}] time for {n_epochs} epochs: {end - start:.2f} s")
    print(f"[{label}] interior loss after training: ", pinn.best_loss["interior"])
    print(f"[{label}] boundary loss after training: ", pinn.best_loss["boundary"])
    print(f"[{label}] total    loss after training: ", pinn.best_loss["total"])
    return key, pinn


def plot_comparison(pinn_new, title_new):
    """Plot the reference PINN of section 1 next to another trained PINN."""
    plot_abstract_approx_spaces(
        (pinn.space, pinn_new.space),  # the approximation spaces
        domain_x,  # the spatial domain
        domain_mu,  # the parameter's domain
        loss=(pinn.losses, pinn_new.losses),  # for plot of the loss: the losses
        residual=(pinn.model, pinn_new.model),  # for plot of the residual: the pde
        error=analytic_solution,  # for plot of the error with respect to a func: the func
        draw_contours=True,
        n_drawn_contours=20,
        titles=("reference", title_new),
    )
    plt.show()

### 2.1/ Number of degrees of freedom: approximation error

The number of degrees of freedom of a PINN is the number of trainable parameters of
its network, set by the sizes of the hidden layers: the richer the approximation
space, the smaller the approximation error.

Build a new PINN with different hidden layer sizes, count its degrees of freedom,
train it with the same collocation points and optimizer as the reference, and
compare:


In [ ]:
key = jax.random.PRNGKey(0)

HIDDEN_SIZES_NEW = [32, 32]

nn_dofs = MLP(
    in_size=INPUT_SIZE, out_size=OUTPUT_SIZE, hidden_sizes=HIDDEN_SIZES_NEW, key=key
)
space_dofs = ApproximationSpace(
    {"x": 2, "mu": 1},  # x has dim 2, mu has dim 1
    [(nn_dofs, "scalar", None)],  # nn_dofs models a scalar function u
    model_type="x_mu",  # u depends on x and mu
)

print("dofs of the reference PINN: ", n_dofs(pinn.space))
print("dofs of the new PINN:       ", n_dofs(space_dofs))

pinn_dofs = Projector(model, space_dofs, sampler, matrix_regularization=1e-5)

key, pinn_dofs = train_and_report(
    key,
    pinn_dofs,
    space_dofs,
    N_EPOCHS,
    N_COLLOC,
    N_BC_COLLOC,
    label=f"hidden sizes {HIDDEN_SIZES_NEW} ({n_dofs(space_dofs)} dofs)",
)

plot_comparison(pinn_dofs, f"hidden sizes {HIDDEN_SIZES_NEW}")

Play with the width and the depth of the hidden layers (e.g. `[8, 8]`,
`[16, 16, 16]`, `[64, 64]`, ...). Which error component decreases when you add
degrees of freedom? Which one limits the error when you add too many?


### 2.2/ Number of collocation points: discretization error

The loss is an integral over the domain, approximated in `Scimba` by a Monte Carlo
sum over `N_COLLOC` interior collocation points and `N_BC_COLLOC` boundary
collocation points (new points are drawn at every epoch). With too few points, the
discrete loss poorly approximates the continuous one: this is the discretization
(quadrature) error.

Train a new PINN, with the same network as the reference, but a different number of
collocation points:


In [ ]:
key = jax.random.PRNGKey(0)

nn_colloc = MLP(in_size=INPUT_SIZE, out_size=OUTPUT_SIZE, hidden_sizes=HIDDEN_SIZES, key=key)
space_colloc = ApproximationSpace(
    {"x": 2, "mu": 1},
    [(nn_colloc, "scalar", None)],
    model_type="x_mu",
)

pinn_colloc = Projector(model, space_colloc, sampler, matrix_regularization=1e-5)

N_COLLOC_NEW = 100
N_BC_COLLOC_NEW = 100

key, pinn_colloc = train_and_report(
    key,
    pinn_colloc,
    space_colloc,
    N_EPOCHS,
    N_COLLOC_NEW,
    N_BC_COLLOC_NEW,
    label=f"N_COLLOC={N_COLLOC_NEW}, N_BC_COLLOC={N_BC_COLLOC_NEW}",
)

plot_comparison(pinn_colloc, f"{N_COLLOC_NEW} interior / {N_BC_COLLOC_NEW} boundary points")

Decrease, then increase the numbers of collocation points. Which error component is
affected? The error saturates once the discretization error becomes negligible
compared to the approximation and optimization errors, and the training time grows
with the number of points.


### 2.3/ Optimizer and number of epochs: optimization error

The training loop only minimizes the loss approximately: after `N_EPOCHS` epochs,
the gap between the loss and its minimum is the optimization error. It depends on
the choice of the optimizer and on the number of epochs.

Several optimizers are implemented in `scimba_jax`:
- Natural gradient descent: [ENG](https://arxiv.org/pdf/2302.13163) and [ANaGRAM](https://arxiv.org/pdf/2412.10782)
- Adam
- L-BFGS
- SS-BFGS and SS-Broyden as described [here](https://www.sciencedirect.com/science/article/pii/S0021999124009045?ref=pdf_download&fr=RR-2&rr=9a3fcedd0bd4d784).

The optimizer must be specified when instantiating the PINN:


In [ ]:
key = jax.random.PRNGKey(0)

nn_opt = MLP(in_size=3, out_size=1, hidden_sizes=[16, 16, 16], key=key)
space_opt = ApproximationSpace(
    {"x": 2, "mu": 1},
    [(nn_opt, "scalar", None)],
    model_type="x_mu",
)

pinn_opt = Projector(
    model,
    space_opt,
    sampler,
    optimizer="SS-BFGS",
)

N_EPOCHS_OPT = 1000

key, pinn_opt = train_and_report(
    key, pinn_opt, space_opt, N_EPOCHS_OPT, N_COLLOC, N_BC_COLLOC, label="chosen optimizer"
)

The valid optimizer names are:
"ENG", "ANaGRAM", "Adam", "L-BFGS", "SS-BFGS" and "SS-Broyden".

Try different optimizers and numbers of epochs. In the plots below, the loss curves
show how fast each optimizer converges; the gap between the final loss and the
plateau is the optimization error.

To compare the trained PINNs on the same figure:


In [ ]:
plot_comparison(pinn_opt, "chosen optimizer")

### 2.4/ Weights of the residuals in the loss: optimization error

The total loss is a weighted sum of the losses of the residuals. Changing the weights
changes the balance the optimizer seeks between them, and with it the optimization
error reached on each residual.

The weights are specified in a dictionary passed to the `Projector` when instantiating the PINN. The keys of the dictionary are the labels of the residuals, and the values are lists of weights (one weight per output dimension of the residual).

The label `"interior"` is given by default to:
- the **interior** of the main geometric domain
- the residual for the **interior** of the main geometric domain
- the loss associated with the residual for the **interior** of the main geometric domain

whereas `"boundary"` is given by default to:
- the **boundary** of the main geometric domain
- the residual for the **boundary** of the main geometric domain
- the loss associated with the residual for the **boundary** of the main geometric domain.

Say one wants to increase the weight of the boundary condition residual in the loss
computation:


In [ ]:
custom_weights = {"interior": [1.0], "boundary": [40.0]}

Next create a new PINN using those weights, train it, and compare it with the
reference:


In [ ]:
key = jax.random.PRNGKey(0)

nn_w = MLP(in_size=INPUT_SIZE, out_size=OUTPUT_SIZE, hidden_sizes=HIDDEN_SIZES, key=key)
space_w = ApproximationSpace(
    {"x": 2, "mu": 1},  # x has dim 2, mu has dim 1
    [(nn_w, "scalar", None)],  # nn_w models a scalar function u
    model_type="x_mu",  # u depends on x and mu
)

pinn_w = Projector(
    model, space_w, sampler, weights=custom_weights, matrix_regularization=1e-5
)

key, pinn_w = train_and_report(
    key, pinn_w, space_w, N_EPOCHS, N_COLLOC, N_BC_COLLOC, label="custom weights"
)

plot_comparison(pinn_w, "custom weights")

### 2.5/ Matrix regularization of the natural gradient: optimization error

The natural gradient optimizers (the default `"ENG"`, but also `"ANaGRAM"`, ...)
solve at each epoch a linear system of the form $(G + \lambda I)\,\delta = g$,
where $G$ is the Gram matrix of the residuals and $g$ the gradient of the loss. The
scalar $\lambda$ is the `matrix_regularization` argument of the `Projector`
(default: `1e-6`): it regularizes $G$, which can be ill-conditioned or even
singular. A large $\lambda$ slows the convergence down; a very small one can make
the linear solve unstable. Changing it changes the optimization error.


In [ ]:
key = jax.random.PRNGKey(0)

nn_reg = MLP(in_size=INPUT_SIZE, out_size=OUTPUT_SIZE, hidden_sizes=HIDDEN_SIZES, key=key)
space_reg = ApproximationSpace(
    {"x": 2, "mu": 1},
    [(nn_reg, "scalar", None)],
    model_type="x_mu",
)

MATRIX_REG = 1e-2

pinn_reg = Projector(model, space_reg, sampler, matrix_regularization=MATRIX_REG)

key, pinn_reg = train_and_report(
    key,
    pinn_reg,
    space_reg,
    N_EPOCHS,
    N_COLLOC,
    N_BC_COLLOC,
    label=f"matrix_regularization={MATRIX_REG}",
)

plot_comparison(pinn_reg, f"matrix_regularization={MATRIX_REG}")

Play with several values of `matrix_regularization` (e.g. `1e-10`, `1e-6`, `1e-2`,
`1.0`). Which error component does it control, and what happens in the extreme
regimes?

Section 2.1 controls the **approximation error**, 2.2 the **discretization error**,
and 2.3--2.5 the **optimization error**.

## 3/ Approximation of the solution of a 1D heat equation

Let us now demonstrate how to define and train a PINN for a time-dependent problem.

We consider the 1D heat equation

$$\partial_t u - \partial_{xx} u = 0 \text{ in } \Omega \times (0, T)$$

with boundary conditions:
$$u = 0 \text{ on } \partial \Omega \times (0, T)$$

and initial conditions:
$$u = u_0 \text{ on } \Omega \times {0}$$

where $u: \Omega \times (0, T) \to \mathbb{R}$ is the unknown function,
$\Omega \subset \mathbb{R}$ and $(0, T) \subset \mathbb{R}$.
Homogeneous Dirichlet boundary conditions are prescribed.

The exact solution reads
$$u(x,t) = \sin(\pi x) \exp(-\pi^2 t),$$
with the initial condition $u_0(x) = \sin(\pi x)$.

### Time and space domains


In [ ]:
from scimba_jax.domains.meshless_domains.domains_1d import Segment1D

domain_t = (0.0, 0.5)
domain_x = Segment1D((0.0, 1.0), is_main_domain=True)

### Sampler on $(0, T) \times \Omega$


In [ ]:
from scimba_jax.nonlinear_approximation.integration.monte_carlo_time import (
    UniformTimeSampler,
)

sampler = TensorizedSampler(
    [
        UniformTimeSampler(domain_t),
        DomainSampler(domain_x),
    ],
    model_type="t_x",
    bc=True,
    ic=True,
)

Notice the optional arguments:
- `model_type` to specify the name and order of variables,
- `bc` to specify that samples are required for the boundary condition,
- `ic` to specify that samples are required for the initial condition.

### Right-hand sides, initial and exact solution

Next we define the initial and the exact solutions. This time, there is no `mu`, as the problem is not parameter-dependent. The framework automatically adapts to that situation.


In [ ]:
def exact_sol(t: jnp.ndarray, x: jnp.ndarray) -> jnp.ndarray:
    """Exact solution of the 1D heat equation."""
    return jnp.exp(-(t * jnp.pi**2)) * jnp.sin(jnp.pi * x)


def f_init(x: jnp.ndarray) -> jnp.ndarray:
    """Initial condition of the 1D heat equation."""
    t = jnp.zeros_like(x)
    return exact_sol(t, x)

### Physical model

The model corresponding to the considered problem is already defined in `scimba_jax`.


In [ ]:
from scimba_jax.physical_models.temporal_pde.heat_equations import HeatND

model = HeatND(
    main_domain=domain_x,
    time_domain=domain_t,
    bc="weak",
    ic="weak",
    f_ic_rhs=lambda *args: f_init(*args),
)

Other possible optional arguments are `f_rhs` and `f_bc_rhs` which are zero functions by default:


In [ ]:
model = HeatND(
    main_domain=domain_x,
    time_domain=domain_t,
    bc="weak",
    ic="weak",
    f_rhs=lambda *args: jnp.zeros_like(args[0]),
    f_bc_rhs=lambda *args: jnp.zeros_like(args[0]),
    f_ic_rhs=f_init,
)

### Approximation space

We wish to approximate the solution with a multi-layer perceptron (MLP) with two inputs ($t$ and $x$), one output ($u(t, x)$) and two hidden layers of $16$ neurons each:


In [ ]:
key = jax.random.PRNGKey(0)  # initialize the random number generator

INPUT_SIZE = 2
OUTPUT_SIZE = 1  # scalar function u
HIDDEN_SIZES = [16, 16]  # two hidden layers with 16 neurons each

nn = MLP(in_size=INPUT_SIZE, out_size=OUTPUT_SIZE, hidden_sizes=HIDDEN_SIZES, key=key)

space = ApproximationSpace(
    {"x": 1},  # here t is implicitly of dimension 1
    [(nn, "scalar", None)],
    model_type="t_x",
)

### PINN training

We finally create a PINN with custom weights:


In [ ]:
custom_weights = {"interior": [1.0], "boundary": [40.0], "ic interior": [100.0]}

pinn = Projector(model, space, sampler, weights=custom_weights)

N_COLLOC = 900
N_BC_COLLOC = 2000
N_IC_COLLOC = 2000
N_EPOCHS = 50

start = timeit.default_timer()
key, pinn = pinn.project(key, space, N_EPOCHS, N_COLLOC, N_BC_COLLOC, N_IC_COLLOC)
end = timeit.default_timer()

plot_abstract_approx_space(
    pinn.space,
    domain_x,
    time_domain=domain_t,
    time_values=[0.0, 0.25, 0.5],
    loss=pinn.losses,
    solution=exact_sol,
    residual=pinn.model,
    error=exact_sol,
    title="learning sol of 1D heat equation with a PINN",
)

Don't hesitate to play with the hyperparameters!